Normal Chain

In [2]:
from json import load
from langchain_google_genai import ChatGoogleGenerativeAI
from langchain_core.prompts import PromptTemplate
from langchain_core.output_parsers import StrOutputParser
from dotenv import load_dotenv

load_dotenv()

llm = ChatGoogleGenerativeAI(model="gemini-3-flash-preview", temperature=0.0)
parser = StrOutputParser()

template = PromptTemplate.from_template("You are a helpful assistant. Answer the question: {question}")

chain = template | llm | parser

result = chain.invoke({"question": "What is the capital of Jordan?"})
print(result)
print("*"*60)

chain.get_graph().print_ascii()

The capital of Jordan is **Amman**.
************************************************************
      +-------------+      
      | PromptInput |      
      +-------------+      
             *             
             *             
             *             
    +----------------+     
    | PromptTemplate |     
    +----------------+     
             *             
             *             
             *             
+------------------------+ 
| ChatGoogleGenerativeAI | 
+------------------------+ 
             *             
             *             
             *             
    +-----------------+    
    | StrOutputParser |    
    +-----------------+    
             *             
             *             
             *             
+-----------------------+  
| StrOutputParserOutput |  
+-----------------------+  


Sequential Chain

In [5]:
from json import load
from langchain_google_genai import ChatGoogleGenerativeAI
from langchain_core.prompts import PromptTemplate
from langchain_core.output_parsers import StrOutputParser
from dotenv import load_dotenv

load_dotenv()

llm = ChatGoogleGenerativeAI(model="gemini-2.5-flash", temperature=0.0)
parser = StrOutputParser()

seq_template1 = PromptTemplate.from_template("""
Give me a short summary on topic : {topic}""")

seq_template2 = PromptTemplate.from_template("""
Based on the summary, give me 5 key pints on it \n {summary}""")

seq_chain = seq_template1 | llm | parser | seq_template2 | llm | parser

seq_result = seq_chain.invoke({"topic": "Generative AI"})

print(seq_result)
print("*"*60)
print("*"*60)
seq_chain.get_graph().print_ascii()

Here are 5 key points based on the summary:

1.  **Core Function:** Generative AI creates new, original content, rather than just analyzing or classifying existing data.
2.  **Learning Mechanism:** It learns patterns, structures, and styles from vast amounts of training data.
3.  **Output Nature:** It generates novel outputs that are similar to, but not identical to, what it was trained on.
4.  **Content Types:** It can produce diverse content, including text, images, audio, and video.
5.  **Primary Applications:** It's used to assist in creative tasks and automate content generation.
************************************************************
************************************************************
      +-------------+      
      | PromptInput |      
      +-------------+      
             *             
             *             
             *             
    +----------------+     
    | PromptTemplate |     
    +----------------+     
             *             
      

Parallel Chain

In [6]:
from langchain_google_genai import ChatGoogleGenerativeAI
from langchain_core.prompts import PromptTemplate
from langchain_core.output_parsers import StrOutputParser
from langchain_core.runnables import RunnableParallel
from dotenv import load_dotenv

load_dotenv()

llm = ChatGoogleGenerativeAI(model="gemini-2.5-flash", temperature=0.0)
parser = StrOutputParser()

par_template1 = PromptTemplate.from_template("""
Give me a short summary on topic : {topic}""")

par_template2 = PromptTemplate.from_template("""
Based on the summary, give me 5 key pints on it \n {topic}""")

par_chain1 = par_template1 | llm | parser
par_chain2 = par_template2 | llm | parser


parallel_chain = RunnableParallel(

  {
    "summary": par_chain1,
    "key_points": par_chain2
  }
)

mer_template = PromptTemplate.from_template("""
  Combine the following information into a comprehensive output:
                                             
  Summary: {summary}
  Key Points: {key_points}

""")

mer_chain = mer_template | llm | parser

final_chain = parallel_chain | mer_chain

par_result = final_chain.invoke({"topic": "Generative AI"})
print(par_result)
print("="*60)
final_chain.get_graph().print_ascii()

Generative AI represents a cutting-edge type of artificial intelligence designed to **create new, original content** rather than merely analyzing or classifying existing data. At its core, it learns intricate patterns, structures, and styles from vast amounts of training data—encompassing text, images, audio, and video. This deep learning enables it to **generate novel outputs** that, while similar to its training data, are distinct and original.

Key aspects of Generative AI include:

*   **Creation of New Content:** Its primary function is to produce novel, original content across diverse modalities, including text (like articles, code, or stories), images, audio, video, and more. This capability moves beyond traditional AI's analytical tasks, venturing into creative generation.
*   **Learning from Data Patterns:** Generative models meticulously learn underlying patterns, structures, and styles from extensive datasets during their training phase. This foundational learning allows the

Conditional Chain

In [9]:
from langchain_google_genai import ChatGoogleGenerativeAI
from langchain_core.prompts import PromptTemplate
from langchain_core.output_parsers import StrOutputParser, PydanticOutputParser
from langchain_core.runnables import RunnableParallel, RunnableBranch
from pydantic import BaseModel, Field
from typing import Literal
from dotenv import load_dotenv

load_dotenv()

class ReviewSentiment(BaseModel):
    sentiment: Literal["positive", "negative", "neutral"] = Field(description="The sentiment of the movie review")

llm = ChatGoogleGenerativeAI(model="gemini-2.5-flash", temperature=0.0)
str_parser = StrOutputParser()
pydantic_parser = PydanticOutputParser(pydantic_object=ReviewSentiment)

# القالب الأول لتحليل المشاعر
con_template1 = PromptTemplate.from_template("""
    Give me sentiment of the movie review {review} \n
    format_instruction: {format_instruction}
""",
partial_variables={"format_instruction": pydantic_parser.get_format_instructions()}
)

review_chain = con_template1 | llm | pydantic_parser

# قوالب الردود حسب المشاعر
pos_template = PromptTemplate.from_template("""
    The review is positive so write a short appreciation: \n {review}
""")

neg_template = PromptTemplate.from_template("""
    The review is negative so write a short critique: \n {review}
""")

neu_template = PromptTemplate.from_template("""
    The review is neutral so write a short balanced remarks: \n {review}
""")

defult_template = PromptTemplate.from_template("""
    I couldn't detect the clear sentiment. Provide a neutral response: \n {review}
""")

# إنشاء RunnableBranch
branch_chain = RunnableBranch(
    (lambda x: x.sentiment == "positive", pos_template | llm | str_parser),
    (lambda x: x.sentiment == "negative", neg_template | llm | str_parser),
    (lambda x: x.sentiment == "neutral", neu_template | llm | str_parser),
    defult_template | llm | str_parser
)

final_con_chain = review_chain | branch_chain

# ✅ طباعة هيكل الـ chain
print("\n" + "="*60)
print("هيكل final_con_chain:")
print("="*60)
final_con_chain.get_graph().print_ascii()

print("\n" + "="*60)
print("هيكل branch_chain:")
print("="*60)
branch_chain.get_graph().print_ascii()

# تجربة التنفيذ
review = "It was okay—some good moments, but nothing particularly memorable. Just an average watch."
result = final_con_chain.invoke({"review": review})
print("\n" + "="*60)
print("النتيجة:")
print("="*60)
print(result)


هيكل final_con_chain:
      +-------------+      
      | PromptInput |      
      +-------------+      
             *             
             *             
             *             
    +----------------+     
    | PromptTemplate |     
    +----------------+     
             *             
             *             
             *             
+------------------------+ 
| ChatGoogleGenerativeAI | 
+------------------------+ 
             *             
             *             
             *             
 +----------------------+  
 | PydanticOutputParser |  
 +----------------------+  
             *             
             *             
             *             
        +--------+         
        | Branch |         
        +--------+         
             *             
             *             
             *             
     +--------------+      
     | BranchOutput |      
     +--------------+      

هيكل branch_chain:
+-------------+  
| PromptInput |